# ACC — version 3.2 : modèle consolidé

Cette version regroupe tous les calculs des versions 3.x dans un **moteur unique**, utilisé à la fois par ce notebook et par l'application `app_acc.py`.

**Fichiers à placer dans le même dossier que ce notebook :**

| Fichier | Rôle |
| --- | --- |
| `acc_donnees.py` | Lecture et contrôle des fichiers Excel (votre module, inchangé) |
| `acc_moteur.py` | Contraintes, bilans énergétiques, gains, optimisation multi-objectif |
| `acc_graphiques.py` | Tous les graphiques |
| `app_acc.py` | Application : `streamlit run app_acc.py` |

**Ce qui change par rapport à la version 3.1 :**

| Version 3.1 | Version 3.2 |
| --- | --- |
| Bloc 1 (chargement + graphique annuel) | Bloc 1 : chargement seul |
| Blocs 2 à 5 (graphiques, producteurs, distances) | Bloc 2 : données et graphiques |
| Paramètres dispersés dans chaque bloc | Bloc 3 : tous les paramètres au même endroit |
| Bloc 8 (boucles admissibles, front à 4 objectifs, Score_global) | Bloc 4 : toutes les boucles, avec énergie **et gains** ; Score_global supprimé |
| Blocs 8, 8 bis, 14 (objectifs figés) | Bloc 5 : objectifs et seuils au choix, front de Pareto, sélection par priorité |
| Blocs 9, 11, 12, 13 (analyse d'une boucle) | Bloc 6 : analyse détaillée de la boucle retenue |
| — | Bloc 7 : comparaison de plusieurs stratégies d'optimisation |

Les résultats énergétiques et économiques sont identiques à ceux des blocs 8, 11, 13 et 14 de la version 3.1.1 (vérifié sur les mêmes calculs).

**Les prix de l'énergie par défaut sont fictifs.** Seul le taux d'accise correspond à un tarif réel (catégorie > 36 kVA, en vigueur depuis le 1er août 2026), à mettre à jour.

In [ ]:
# ==============================================================
# BLOC 1 — CHARGEMENT DES DONNÉES
# ==============================================================
# acc_donnees.py lit, contrôle et synchronise les fichiers Excel (inchangé).
# acc_moteur.depuis_jeu_acc() prépare ensuite les données pour tous les calculs.

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from acc_donnees import ConfigurationACC, charger_donnees_acc
import acc_moteur as m
import acc_graphiques as g

DOSSIER_BATIMENTS = Path("donnees/batiments")
PAS_MINUTES = 60                  # 15 pour de vraies données au quart d'heure
FUSEAU_HORAIRES_NAIFS = "UTC"     # "Europe/Paris" si les fichiers sont en heure locale
CONTROLE_PV = "avertir"           # "erreur" pour des données validées

jeu_acc = charger_donnees_acc(ConfigurationACC(
    dossier=DOSSIER_BATIMENTS,
    pas_minutes=PAS_MINUTES,
    fuseau_horaires_naifs=FUSEAU_HORAIRES_NAIFS,
    strict=True,
    controle_pv=CONTROLE_PV,
))
jeu_acc.exiger_complet()
donnees = m.depuis_jeu_acc(jeu_acc)


def montrer(fig):
    """Affiche une figure à l'endroit voulu de la sortie."""
    display(fig)
    plt.close(fig)


print(f"Bâtiments : {donnees.n} (dont {int(donnees.est_producteur.sum())} producteurs PV)")
print(f"Pas de mesure : {jeu_acc.pas_minutes} min")
print(f"Période : {jeu_acc.debut} → {jeu_acc.fin}")
print("\n=== RAPPORT DE SYNCHRONISATION ===")
print(jeu_acc.rapport.to_string(index=False))

In [ ]:
# ==============================================================
# BLOC 2 — DONNÉES ET PREMIERS GRAPHIQUES
# ==============================================================

# Bâtiment détaillé dans les cartes de chaleur (par défaut : premier producteur)
BATIMENT_DETAIL = int(donnees.numeros[donnees.est_producteur][0]) \
    if donnees.est_producteur.any() else int(donnees.numeros[0])
D_MAX_CARTE_KM = 10.0             # liaisons tracées sur la carte

bilan = m.bilan_batiments(donnees)
print("=== BILAN ANNUEL PAR BÂTIMENT ===")
display(bilan.round(2))

montrer(g.fig_bilan_batiments(bilan))
montrer(g.fig_productible(bilan))
montrer(g.fig_profil_mensuel(donnees))
montrer(g.fig_profil_journalier(donnees))
for variable in ["Consommation", "Production"]:
    montrer(g.fig_carte_chaleur(donnees, BATIMENT_DETAIL, variable))

noms = [f"B{b}" for b in donnees.numeros]
print("=== DISTANCES ENTRE BÂTIMENTS (km) ===")
display(pd.DataFrame(donnees.distances, index=noms, columns=noms).round(2))
montrer(g.fig_carte(donnees, D_MAX_CARTE_KM))

In [ ]:
# ==============================================================
# BLOC 3 — PARAMÈTRES (UN SEUL ENDROIT)
# ==============================================================

# Contraintes de constitution d'une boucle (hypothèses du modèle)
CONTRAINTES = m.Contraintes(
    d_max_km=10.0,                    # distance max entre deux participants
    p_pv_max_kwc=500.0,               # puissance PV totale STRICTEMENT inférieure
    nb_max_batiments=0,               # 0 = sans limite
    exiger_consommateur_sans_pv=True,
)

# Plages tarifaires en heure locale
PLAGES = m.PlagesHoraires(
    creuses_debut=22, creuses_fin=6,
    pointe_debut=18, pointe_fin=20, pointe_jours_ouvres=True,
)

# Prix en €/kWh hors TVA — PRIX DE L'ÉNERGIE FICTIFS
TARIFS = m.Tarifs(
    prix_fournisseur={"Creuses": 0.10, "Pleines": 0.13, "Pointe": 0.13},  # part énergie seule
    prix_acc={"Creuses": 0.10, "Pleines": 0.10, "Pointe": 0.10},
    valeur_surplus=0.0,               # valeur du surplus pour le producteur sans ACC
    taux_accise=0.02635,              # > 36 kVA depuis le 1er août 2026 (0.03062 si ≤ 36 kVA)
    seuil_accise_kwc=1000.0,          # accise nulle si installation < 1 MWc
    tva_non_recuperable=0.0,          # 0.20 pour des participants qui ne récupèrent pas la TVA
    frais_par_participant=0.0,        # €/an
)

CONTRAINTES.verifier()
TARIFS.verifier()
periodes = m.classer_periodes(donnees.index, PLAGES)

print("Pas de temps par plage tarifaire :")
print(pd.Series(periodes).value_counts().reindex(list(m.PERIODES), fill_value=0).to_string())

In [ ]:
# ==============================================================
# BLOC 4 — TOUTES LES BOUCLES ADMISSIBLES : ÉNERGIE ET GAINS
# ==============================================================
# Ancien Bloc 8 : mêmes contraintes, mêmes boucles, mêmes noms ACC1, ACC2…
# Ajouts : gain total annuel, gain moyen, maximal et minimal par participant.
# Suppression : Score_global (pondération arbitraire).

EXPORTER = True
DOSSIER_SORTIE = Path("resultats_ACC")

boucles = m.enumerer_boucles(donnees, CONTRAINTES)
tableau = m.evaluer_boucles(donnees, boucles, TARIFS, periodes)
print(f"Boucles admissibles : {len(tableau)}")

COLONNES_TABLEAU = [
    "Solution", "Participants", "Nb_batiments", "Nb_producteurs",
    "Puissance_PV_kWc", "Distance_max_km", "Surplus_MWh", "E_ACC_MWh",
    "TAC_pct", "TAP_pct",
    "Valeur_collective_EUR", "Gain_moyen_EUR", "Gain_max_EUR", "Gain_min_EUR",
    "Participant_gain_min",
]
with pd.option_context("display.max_rows", 500, "display.max_columns", None):
    display(tableau[COLONNES_TABLEAU].round(2))

montrer(g.fig_meilleur_par_taille(
    tableau, ["E_ACC_MWh", "Valeur_collective_EUR", "Gain_moyen_EUR", "Gain_min_EUR"]))

if EXPORTER:
    DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
    fichier = DOSSIER_SORTIE / "Boucles_ACC_admissibles.csv"
    tableau.drop(columns=["Indices"]).to_csv(fichier, sep=";", index=False,
                                              encoding="utf-8-sig")
    print("Tableau complet enregistré :", fichier.resolve())

In [ ]:
# ==============================================================
# BLOC 5 — OPTIMISATION MULTI-OBJECTIF AU CHOIX
# ==============================================================
# 1. Seuils : contraintes sur les indicateurs (ex. TAC ≥ 60 %, personne ne perd).
# 2. Front de Pareto des objectifs choisis.
# 3. Sélection par priorité sur ce front : on garde les boucles à moins de
#    TOLERANCE_PCT du meilleur sur l'objectif 1, puis on optimise l'objectif 2…
# Aucune pondération : l'ordre de la liste OBJECTIFS fixe les priorités.

print("Indicateurs disponibles (objectifs ou seuils) :")
for colonne, (nom, unite, sens) in m.INDICATEURS.items():
    print(f"  {colonne:<26} {nom} ({unite or '-'}) — sens habituel : {sens}")

OBJECTIFS = [
    m.Objectif("Valeur_collective_EUR", "max"),
    m.Objectif("Gain_min_EUR", "max"),
]
SEUILS = [
    m.Seuil("Gain_min_EUR", ">=", 0.0),     # aucun participant ne perd d'argent
    m.Seuil("TAC_pct", ">=", 60.0),         # au moins 60 % du surplus valorisé
]
TOLERANCE_PCT = 5.0
AXES_GRAPHIQUE = None   # None : deux premiers objectifs ; sinon ("E_ACC_MWh", "TAC_pct")

resultat = m.optimiser(tableau, OBJECTIFS, SEUILS, TOLERANCE_PCT)
solution = resultat.solution

print(f"\nBoucles respectant les seuils : {len(resultat.candidats)}")
print(f"Boucles non dominées : {int(resultat.candidats['Pareto'].sum())}")
print("\n=== ÉTAPES DE LA SÉLECTION PAR PRIORITÉ ===")
display(resultat.etapes.round(2))

print(f"\n=== BOUCLE RETENUE : {solution['Solution']} — {solution['Participants']} ===")
display(solution[COLONNES_TABLEAU].to_frame("Valeur"))

front = resultat.candidats[resultat.candidats["Pareto"]].sort_values(
    OBJECTIFS[0].colonne, ascending=OBJECTIFS[0].sens == "min")
print("\n=== FRONT DE PARETO ===")
display(front[COLONNES_TABLEAU].round(2))

axes = AXES_GRAPHIQUE or ([o.colonne for o in OBJECTIFS] + ["TAC_pct"])[:2]
montrer(g.fig_nuage_objectifs(resultat.candidats, axes[0], axes[1], solution,
                              tableau_complet=tableau))

In [ ]:
# ==============================================================
# BLOC 6 — ANALYSE DÉTAILLÉE DE LA BOUCLE RETENUE
# ==============================================================

BOUCLE_A_ANALYSER = None   # None : boucle retenue au Bloc 5 ; sinon un nom, ex. "ACC16"

ligne = solution if BOUCLE_A_ANALYSER is None \
    else tableau.set_index("Solution").loc[BOUCLE_A_ANALYSER]
nom_boucle = solution["Solution"] if BOUCLE_A_ANALYSER is None else BOUCLE_A_ANALYSER
numeros_boucle = m.numeros_de(ligne)
analyse = m.analyser_boucle(donnees, numeros_boucle, TARIFS, periodes)
r = analyse["resume"]

print(f"=== {nom_boucle} : {r['Participants']} ===")
print(f"Énergie partagée : {r['E_ACC_MWh']:.1f} MWh/an | TAC {r['TAC_pct']:.1f} % | "
      f"TAP {r['TAP_pct']:.1f} % | adéquation temporelle {r['Adequation_pct']:.1f} %")
print(f"Surplus non valorisé : {r['Surplus_non_valorise_MWh']:.1f} MWh/an | "
      f"achat complémentaire : {r['Achat_complementaire_MWh']:.1f} MWh/an")
print(f"Gain total : {r['Valeur_collective_EUR']:,.0f} €/an | gain moyen : "
      f"{r['Gain_moyen_EUR']:,.0f} €/an | gain minimal : {r['Gain_min_EUR']:,.0f} €/an "
      f"({r['Participant_gain_min']})".replace(",", " "))

print("\n=== GAINS PAR PARTICIPANT (€/an) ===")
display(analyse["participants"].round(1))
montrer(g.fig_gains_participants(analyse["participants"],
                                 f"{nom_boucle} — gain annuel par participant"))
montrer(g.fig_bilan_mensuel_boucle(analyse["series"]))
montrer(g.fig_profil_journalier_boucle(analyse["series"]))
print("\n=== BILAN PAR PLAGE TARIFAIRE (MWh/an) ===")
display(analyse["par_periode"].round(2))
montrer(g.fig_par_periode(analyse["par_periode"]))
montrer(g.fig_carte(donnees, CONTRAINTES.d_max_km, numeros_boucle,
                    f"Localisation de {nom_boucle}"))

In [ ]:
# ==============================================================
# BLOC 7 — COMPARER PLUSIEURS STRATÉGIES D'OPTIMISATION
# ==============================================================
# Chaque stratégie : (liste d'objectifs par priorité, liste de seuils).
# Ajouter ou retirer des lignes librement.

STRATEGIES = {
    "Énergie partagée, TAC ≥ 60 %": (
        [m.Objectif("E_ACC_MWh", "max")], [m.Seuil("TAC_pct", ">=", 60.0)]),
    "Gain total": ([m.Objectif("Valeur_collective_EUR", "max")], []),
    "Gain moyen par participant": ([m.Objectif("Gain_moyen_EUR", "max")], []),
    "Gain minimal, puis gain total": (
        [m.Objectif("Gain_min_EUR", "max"), m.Objectif("Valeur_collective_EUR", "max")], []),
    "TAC puis TAP (ancien Bloc 8 bis)": (
        [m.Objectif("TAC_pct", "max"), m.Objectif("TAP_pct", "max")], []),
    "Choix du Bloc 5": (OBJECTIFS, SEUILS),
}

lignes = {}
for nom, (objectifs, seuils) in STRATEGIES.items():
    try:
        lignes[nom] = m.optimiser(tableau, objectifs, seuils, TOLERANCE_PCT).solution
    except ValueError as erreur:
        print(f"{nom} : {erreur}")

comparaison = pd.DataFrame(lignes).T
print("=== BOUCLE CHOISIE PAR CHAQUE STRATÉGIE ===")
display(comparaison[COLONNES_TABLEAU].round(2))

# Avec un même prix pour tous, le gain total est proportionnel à l'énergie
# partagée : ces deux objectifs sont alors redondants (points alignés).
montrer(g.fig_comparaison_strategies(tableau, comparaison, "E_ACC_MWh", "Valeur_collective_EUR"))
# Le gain minimal crée un vrai compromis : le maximiser seul favorise les
# petites boucles, car exclure le moins favorisé relève le minimum.
montrer(g.fig_comparaison_strategies(tableau, comparaison, "Valeur_collective_EUR", "Gain_min_EUR"))